# Classification Example

## Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import StandardScaler

# Metrics
from sklearn.metrics import balanced_accuracy_score, accuracy_score, precision_score, recall_score, make_scorer

# Validation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

## Load Data

In [2]:
# Load the dataset directly from seaborn's datasets
df = sns.load_dataset('titanic')
df = df[['age', 'fare', 'embarked', 'sibsp', 'class', 'deck', 'alive']]
df.head()

,age,fare,embarked,sibsp,class,deck,alive
0,22.0,7.2500,S,1,Third,NaN,no
1,38.0,71.2833,C,1,First,C,yes
2,26.0,7.9250,S,0,Third,NaN,yes
3,35.0,53.1000,S,1,First,C,yes
4,35.0,8.0500,S,0,Third,NaN,no


In [ ]:
df.shape

In [ ]:
df['alive'].value_counts()

## Feature Eng. Pipeline

In [ ]:
# Canvia directament el target
df['alive'] = df['alive'].map({'yes': 1, 'no': 0})

In [ ]:
# Cada Pipeline és una "recepta" de passos. Encara NO toquem les dades.

# Numèriques: només imputem amb la mediana
median_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

# sibsp decidim imputar nuls amb 0
sibsp_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
])

# Categòriques SENSE ordre: imputem amb la moda + One-Hot
embarked_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')),
])

# Categòrica AMB ordre: imputem amb la moda + Ordinal (indicant l'ordre)
class_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(categories=[['Third', 'Second', 'First']])),
])

# deck té molts nuls: imputem amb un valor constant "Unknown" + One-Hot
deck_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')),
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('median', median_pipeline, ['age', 'fare']),
        ('sibsp', sibsp_pipeline, ['sibsp']),
        ('embarked', embarked_pipeline, ['embarked']),
        ('class', class_pipeline, ['class']),
        ('deck', deck_pipeline, ['deck']),
    ],
    remainder='passthrough',  # deixa la resta de features tal i com estan
)

preprocessor

## Train / Test

In [ ]:
X = df.drop('alive', axis=1)
y = df['alive']

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

## Cross-validation

In [ ]:
# Declare KFold
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [ ]:
# Declare scores to be used
scoring = {
    'Acc': make_scorer(accuracy_score),
    'Prec': make_scorer(precision_score),
    'Rec': make_scorer(recall_score)
}

In [ ]:
def print_metrics(cv_results):
    for sc in scoring.keys():
        print(f'Train {sc}:', cv_results[f'train_{sc}'].mean().round(2))
    print()
    for sc in scoring.keys():
        print(f'Validation {sc}:', cv_results[f'test_{sc}'].mean().round(2))

## Baseline

In [ ]:
from sklearn.dummy import DummyClassifier

In [ ]:
bl = DummyClassifier(strategy='stratified')

bl_cv = cross_validate(
    bl,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(bl_cv)

## Logistic Regression

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=10_000))
])

lr_cv = cross_validate(
    lr_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(lr_cv)

## Decision Tree

In [ ]:
from sklearn.tree import plot_tree

In [ ]:
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('dt', DecisionTreeClassifier(max_depth=2))
])

dt_cv = cross_validate(
    dt_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

print_metrics(dt_cv)

In [ ]:
dt_pipeline.fit(X_train, y_train)
plt.figure(figsize=(20,10))
plot_tree(
    dt_pipeline['dt'],
    filled=True,
    feature_names=dt_pipeline['preprocessor'].get_feature_names_out(),
    class_names=['0', '1'],
    rounded=True
)
plt.show()

### Confusion Matrix Example

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('dt', DecisionTreeClassifier(max_depth=5))
])

# cross_val_predict returns the predictions for each data point in the validation sets
val_preds = cross_val_predict(dt_pipeline, X_train, y_train, cv=kf)

cm = confusion_matrix(y_train, val_preds, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[0, 1]
)
disp.plot()
plt.show()

### Predicted Probabilities

`cross_val_predict()` function performs cross-validation and returns the predictions for each instance in the training set as if each instance were in a validation set:
- It splits the training data into multiple "folds" (validation sets).
- For each fold, it trains the model on the other folds (excluding the validation set).
- It then uses the model to make predictions on the validation set.
- Result: You get predictions for the entire training set, but importantly, the model never sees the instances it's predicting (they were always held out for validation during training).
- Returning probabilities: The `cross_val_predict()` function has an option to return class probabilities instead of just class predictions. You can enable this by setting `method="predict_proba"` in the function call. This provides the probabilities of each class for each instance, which can be useful when you want to assess the model's confidence in its predictions.

In [ ]:
# This provides predictions for the entire training set, with each prediction made on data the model hasn’t seen during training
y_pred = cross_val_predict(dt_pipeline, X_train, y_train, cv=kf)
y_pred

In [ ]:
# This provides predicted probabilities for each class and instance on the entire training set,
# with each prediction made on data the model hasn’t seen during training
# For each row, it gives 2 probabilities: probability of class 0 and probability of class 1
y_pred_prob = cross_val_predict(dt_pipeline, X_train, y_train, cv=kf, method='predict_proba')
y_pred_prob

In [ ]:
# Probability of class 1
y_prob1 = y_pred_prob[:, 1]
y_prob1

In [ ]:
# Prediction with threshold for the probability of class 1 > 80%
(y_prob1 >= 0.8).astype(int)

In [ ]:
# With `predict` method, the decision threshold is 50% by default
dt_pipeline.fit(X_train, y_train)
print('Precision:', round(precision_score(y_test, dt_pipeline.predict(X_test)), 2))
print('Recall:', round(recall_score(y_test, dt_pipeline.predict(X_test)), 2))

In [ ]:
y_test_prob1 = dt_pipeline.predict_proba(X_test)[:, 1]
d = {}

# Sweep across different threshold values and check how this affects precision and recall
for th in np.arange(0, 1.1, 0.1):
    y_test_pred = (y_test_prob1 >= th).astype(int)
    d[th] = {
        'precision': round(precision_score(y_test, y_test_pred), 2),
        'recall': round(recall_score(y_test, y_test_pred), 2)
    }

In [ ]:
pd.DataFrame(d)